In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, confusion_matrix
)

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline


In [5]:
df = pd.read_csv('/Users/ishaanjain/DL PROJECT/data_processed/ilpd_cleaned.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nClass balance:\n", df["liver_disease"].value_counts())
print("\nNulls per column:\n", df.isnull().sum())


Shape: (583, 11)

Columns: ['age', 'gender', 'total_bilirubin', 'direct_bilirubin', 'alkaline_phosphotase', 'alt', 'ast', 'total_proteins', 'albumin', 'albumin_globulin_ratio', 'liver_disease']

Class balance:
 liver_disease
1    416
0    167
Name: count, dtype: int64

Nulls per column:
 age                       0
gender                    0
total_bilirubin           0
direct_bilirubin          0
alkaline_phosphotase      0
alt                       0
ast                       0
total_proteins            0
albumin                   0
albumin_globulin_ratio    0
liver_disease             0
dtype: int64


In [6]:
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Removed {before - after} duplicate rows. Final rows: {after}")


Removed 13 duplicate rows. Final rows: 570


In [7]:
X = df.drop("liver_disease", axis=1)
y = df["liver_disease"]


In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

def evaluate_model(name, model):
    print(f"\n=== {name} ===")
    results = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    for metric in scoring.keys():
        scores = results[f"test_{metric}"]
        print(f"{metric}: {scores.mean():.4f} ± {scores.std():.4f}")


In [9]:
rf_model = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42)),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=7,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=42
    ))
])

evaluate_model("RandomForest + SMOTE", rf_model)



=== RandomForest + SMOTE ===
accuracy: 0.6526 ± 0.0371
precision: 0.8330 ± 0.0423
recall: 0.6430 ± 0.0349
f1: 0.7249 ± 0.0295
roc_auc: 0.7128 ± 0.0335


In [10]:
gb_model = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42)),
    ("gb", GradientBoostingClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

evaluate_model("GradientBoosting + SMOTE", gb_model)



=== GradientBoosting + SMOTE ===
accuracy: 0.6632 ± 0.0326
precision: 0.7844 ± 0.0350
recall: 0.7290 ± 0.0228
f1: 0.7552 ± 0.0216
roc_auc: 0.6913 ± 0.0660


In [11]:
svm_model = ImbPipeline(steps=[
    ("smote", SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("svm", SVC(
        kernel="rbf",
        C=5.0,
        gamma="scale",
        probability=True,
        random_state=42
    ))
])

evaluate_model("SVM RBF + SMOTE", svm_model)



=== SVM RBF + SMOTE ===
accuracy: 0.6404 ± 0.0426
precision: 0.8511 ± 0.0549
recall: 0.6036 ± 0.0480
f1: 0.7045 ± 0.0380
roc_auc: 0.7101 ± 0.0411


In [12]:
best_model = svm_model  # change if RF/GB is better

y_true_all = []
y_pred_all = []
y_proba_all = []

for train_idx, test_idx in cv.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)
    y_proba = best_model.predict_proba(X_test)[:, 1]
    
    y_true_all.extend(y_test.tolist())
    y_pred_all.extend(y_pred.tolist())
    y_proba_all.extend(y_proba.tolist())

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)
y_proba_all = np.array(y_proba_all)

cm = confusion_matrix(y_true_all, y_pred_all)
acc = accuracy_score(y_true_all, y_pred_all)
prec = precision_score(y_true_all, y_pred_all)
rec = recall_score(y_true_all, y_pred_all)
f1 = f1_score(y_true_all, y_pred_all)
auc = roc_auc_score(y_true_all, y_proba_all)

print("Confusion matrix:\n", cm)
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")


Confusion matrix:
 [[120  44]
 [161 245]]
Accuracy : 0.6404
Precision: 0.8478
Recall   : 0.6034
F1-score : 0.7050
ROC-AUC  : 0.7050
